# 8パズルと探索木（DFS / BFS）

8パズルを状態空間探索の題材として、深さ優先探索（DFS）と幅優先探索（BFS）を実装する。

- **状態**: 長さ9の `tuple`（`0` が空白タイル）
- **操作**: 空白と上下左右の隣接タイルを入れ替える
- **BFS**: 最短手数の解を保証
- **DFS**: 解は見つかるが最短とは限らない

## Input

盤面は行優先で9要素。`0` が空白。

```
1 2 3
4 5 6
7 8 0
```

→ `(1, 2, 3, 4, 5, 6, 7, 8, 0)`

In [ ]:
INITIAL_STATE = (
    1, 2, 3,
    4, 5, 6,
    0, 7, 8,
)

GOAL_STATE = (
    1, 2, 3,
    4, 5, 6,
    7, 8, 0,
)

In [ ]:
from collections import deque
from typing import Literal

State = tuple[int, ...]
Strategy = Literal["bfs", "dfs"]

## 問題定義

In [ ]:
def print_board(state: State, title: str = "") -> None:
    if title:
        print(title)
    for row in range(3):
        cells = state[row * 3 : (row + 1) * 3]
        print(" ".join(str(c) if c != 0 else "_" for c in cells))
    print()


def is_goal(state: State, goal: State) -> bool:
    return state == goal


def get_successors(state: State) -> list[State]:
    board = list(state)
    blank = board.index(0)
    row, col = divmod(blank, 3)
    successors: list[State] = []

    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = row + dr, col + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            next_board = board.copy()
            next_blank = nr * 3 + nc
            next_board[blank], next_board[next_blank] = (
                next_board[next_blank],
                next_board[blank],
            )
            successors.append(tuple(next_board))

    return successors


def count_inversions(state: State) -> int:
    tiles = [tile for tile in state if tile != 0]
    inversions = 0
    for i, left in enumerate(tiles):
        for right in tiles[i + 1 :]:
            if left > right:
                inversions += 1
    return inversions


def is_solvable(initial: State, goal: State) -> bool:
    """3x3 8パズル: 逆転数 + 空白行（下から数えて）の偶奇が一致すれば解ける。"""
    blank_row_from_bottom = 2 - (initial.index(0) // 3)
    parity = (count_inversions(initial) + blank_row_from_bottom) % 2
    goal_blank_row_from_bottom = 2 - (goal.index(0) // 3)
    goal_parity = (count_inversions(goal) + goal_blank_row_from_bottom) % 2
    return parity == goal_parity

In [ ]:
print_board(INITIAL_STATE, "初期状態")
print_board(GOAL_STATE, "ゴール")
print(f"解ける配置か: {is_solvable(INITIAL_STATE, GOAL_STATE)}")

## 探索

In [ ]:
def reconstruct_path(parent: dict[State, State | None], goal: State) -> list[State]:
    path: list[State] = []
    current: State | None = goal
    while current is not None:
        path.append(current)
        current = parent[current]
    path.reverse()
    return path


def search(
    initial: State,
    goal: State,
    strategy: Strategy = "bfs",
) -> tuple[list[State] | None, int, int]:
    """戻り値: (経路, 展開ノード数, 手数)"""
    if initial == goal:
        return [initial], 0, 0

    if not is_solvable(initial, goal):
        return None, 0, -1

    frontier: deque[State] | list[State]
    if strategy == "bfs":
        frontier = deque([initial])
    else:
        frontier = [initial]

    visited: set[State] = {initial}
    parent: dict[State, State | None] = {initial: None}
    expanded = 0

    while frontier:
        if strategy == "bfs":
            current = frontier.popleft()
        else:
            current = frontier.pop()

        expanded += 1

        if is_goal(current, goal):
            path = reconstruct_path(parent, current)
            return path, expanded, len(path) - 1

        for successor in get_successors(current):
            if successor in visited:
                continue
            visited.add(successor)
            parent[successor] = current
            frontier.append(successor)

    return None, expanded, -1

## 実行

In [ ]:
def show_result(label: str, path: list[State] | None, expanded: int, moves: int) -> None:
    print(f"=== {label} ===")
    if path is None:
        print("解なし（到達不能）")
        return

    print(f"手数: {moves}")
    print(f"展開ノード数: {expanded}")
    print("経路:")
    for step, state in enumerate(path):
        print(f"--- step {step} ---")
        print_board(state)


bfs_path, bfs_expanded, bfs_moves = search(INITIAL_STATE, GOAL_STATE, strategy="bfs")
show_result("BFS", bfs_path, bfs_expanded, bfs_moves)

dfs_path, dfs_expanded, dfs_moves = search(INITIAL_STATE, GOAL_STATE, strategy="dfs")
show_result("DFS", dfs_path, dfs_expanded, dfs_moves)

## 比較

In [ ]:
print(f"{'手法':<6} {'手数':>6} {'展開ノード数':>12}")
print("-" * 28)
print(f"{'BFS':<6} {bfs_moves:>6} {bfs_expanded:>12}")
print(f"{'DFS':<6} {dfs_moves:>6} {dfs_expanded:>12}")

if bfs_path and dfs_path:
    if bfs_moves <= dfs_moves:
        print("\nBFS は最短手数。DFS は最短とは限らない。")
    else:
        print("\nこの例では DFS の方が短い手数になった（DFS は最短を保証しない）。")